# PSM en Python — Econometría Avanzada
**Ana María Díaz Escobar — Pontificia Universidad Javeriana**

Equivalente Python del do-file `16_stata.do`  
Base: `base6.dta` | Tratamiento: `D` | Resultado: `y2`

Instalar dependencias:
```bash
pip install pandas numpy matplotlib seaborn scipy statsmodels pyreadstat
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

# Estilo de gráficos
sns.set_theme(style='whitegrid', palette='colorblind')
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Cargar datos

In [ ]:
import pyreadstat

# Ajusta la ruta a tu carpeta de trabajo
df, meta = pyreadstat.read_dta('base6.dta')

covs = ['personas', 'orden_n', 'ocupado_jefe', 'educa_jefe',
        'ingresos_hogar_jefe', 'hombre']

print(df.shape)
df[['D', 'y1', 'y2'] + covs].describe()

## 2. Estadísticas descriptivas por grupo

In [ ]:
print("Medias por grupo de tratamiento:")
df.groupby('D')[['y1', 'y2'] + covs].mean().T

## 3. Modelo de selección: Logit (equivalente a dprobit)

In [ ]:
formula_ps = 'D ~ ' + ' + '.join(covs)
ps_model = smf.logit(formula_ps, data=df).fit()
print(ps_model.summary())

# Efectos marginales (equivalente a dprobit)
mfx = ps_model.get_margeff()
print(mfx.summary())

## 4. Propensity Score: estimación y soporte común

In [ ]:
df['ps1'] = ps_model.predict()

fig, ax = plt.subplots(figsize=(8, 4))
for d, label, color in [(1, 'Tratados (D=1)', 'steelblue'),
                         (0, 'Controles (D=0)', 'tomato')]:
    df.loc[df['D'] == d, 'ps1'].plot.kde(ax=ax, label=label,
                                          color=color, linewidth=2)
ax.set_xlabel('Propensity Score estimado')
ax.set_ylabel('Densidad')
ax.set_title('Distribución del PS: soporte común')
ax.legend()
plt.tight_layout()
plt.show()

print("\nPS por grupo:")
df.groupby('D')['ps1'].describe()

## 5. PSM: Vecino Más Cercano con `scikit-learn` + emparejamiento manual

Python no tiene un equivalente directo de `psmatch2`. Implementamos NN(1) con reemplazo.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def psm_nn(df, ps_col='ps1', treat_col='D', outcome_col='y2',
           n_neighbors=1, replace=True, common_support=True):
    """
    PSM Vecino Más Cercano.
    Devuelve el ATT estimado y la diferencia estandarizada de covariables.
    """
    treated  = df[df[treat_col] == 1].copy().reset_index(drop=True)
    controls = df[df[treat_col] == 0].copy().reset_index(drop=True)

    # Soporte común
    if common_support:
        ps_min = max(treated[ps_col].min(), controls[ps_col].min())
        ps_max = min(treated[ps_col].max(), controls[ps_col].max())
        treated  = treated[(treated[ps_col]  >= ps_min) & (treated[ps_col]  <= ps_max)]
        controls = controls[(controls[ps_col] >= ps_min) & (controls[ps_col] <= ps_max)]

    # Orden aleatorio (evita dependencia del orden para NN sin reemplazo)
    np.random.seed(50)
    treated = treated.sample(frac=1).reset_index(drop=True)

    # Ajustar NearestNeighbors
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
    nn.fit(controls[[ps_col]])
    distances, indices = nn.kneighbors(treated[[ps_col]])

    # Calcular ATT
    y_treated  = treated[outcome_col].values
    y_matched  = np.array([controls.iloc[idx][outcome_col].mean()
                           for idx in indices])
    att = (y_treated - y_matched).mean()
    se  = (y_treated - y_matched).std() / np.sqrt(len(y_treated))

    return {
        'ATT': att, 'SE': se,
        'n_treated': len(treated), 'n_controls': len(controls)
    }

# NN(1) sin reemplazo con soporte común
result_nn1 = psm_nn(df, n_neighbors=1, replace=False, common_support=True)
print("NN(1) sin reemplazo, soporte común:")
print(f"  ATT = {result_nn1['ATT']:.4f}  SE = {result_nn1['SE']:.4f}")
print(f"  Tratados: {result_nn1['n_treated']}  Controles: {result_nn1['n_controls']}")

# NN(5)
result_nn5 = psm_nn(df, n_neighbors=5, replace=True, common_support=True)
print("\nNN(5) con reemplazo, soporte común:")
print(f"  ATT = {result_nn5['ATT']:.4f}  SE = {result_nn5['SE']:.4f}")

## 6. Balance de covariables: diferencia estandarizada

In [ ]:
def standardized_bias(df, covs, treat_col='D'):
    """Calcula la diferencia estandarizada (equivalente a %Bias de pstest)."""
    treated  = df[df[treat_col] == 1]
    controls = df[df[treat_col] == 0]
    results = []
    for cov in covs:
        mt, mc = treated[cov].mean(), controls[cov].mean()
        vt, vc = treated[cov].var(),  controls[cov].var()
        sb = (mt - mc) / np.sqrt((vt + vc) / 2) * 100
        results.append({'Variable': cov, 'Media Tratados': mt,
                         'Media Controles': mc, '%Bias': sb})
    return pd.DataFrame(results).set_index('Variable')

print("Balance ANTES del matching:")
print(standardized_bias(df, covs).round(2))
print("\nRegla: |%Bias| < 20% es aceptable; < 5% es excelente")

## 7. IPW (Inverse Probability Weighting)

In [ ]:
# Ponderadores ATT: w=1 para tratados, w=ps/(1-ps) para controles
df['w_att'] = np.where(df['D'] == 1, 1.0, df['ps1'] / (1 - df['ps1']))

# Estimación IPW del ATT
ipw_model = smf.wls('y2 ~ D', data=df, weights=df['w_att']).fit()
print("IPW ATT:")
print(ipw_model.summary().tables[1])

## 8. Tabla comparativa de estimadores

In [ ]:
# Diferencia cruda
crude = smf.ols('y2 ~ D', data=df).fit()

# OLS con covariables
ols_ctrl = smf.ols('y2 ~ D + ' + ' + '.join(covs), data=df).fit()

comparacion = pd.DataFrame({
    'Metodo': ['Diferencia cruda', 'OLS con covariables',
               'NN(1) sin reemplazo', 'NN(5) con reemplazo', 'IPW ATT'],
    'ATT': [
        crude.params['D'],
        ols_ctrl.params['D'],
        result_nn1['ATT'],
        result_nn5['ATT'],
        ipw_model.params['D']
    ]
})

print(comparacion.to_string(index=False))

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(comparacion['Metodo'], comparacion['ATT'], color='steelblue', alpha=0.8)
ax.axvline(comparacion.iloc[1]['ATT'], color='red', linestyle='--', label='OLS referencia')
ax.set_xlabel('Estimación del ATT')
ax.set_title('Comparación de estimadores bajo CIA')
ax.legend()
plt.tight_layout()
plt.show()